In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import BinaryLabelDatasetMetric

# Same data but with ethical considerations
data = {
    'years_experience': [5, 8, 3, 6, 2, 7, 4, 9],
    'education_level': [3, 4, 2, 4, 1, 3, 2, 4],
    'gender': ['male', 'female', 'female', 'male', 'female', 'male', 'female', 'male'],
    'hired': [1, 0, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

# Convert gender to binary before creating BinaryLabelDataset
df['gender'] = df['gender'].map({'male': 1, 'female': 0}) # This line is moved up

# Convert to AIF360 dataset format
dataset = BinaryLabelDataset(
    df=df,
    label_names=['hired'],
    protected_attribute_names=['gender'],
    favorable_label=1,
    unfavorable_label=0
)

# Check bias before mitigation
metric = BinaryLabelDatasetMetric(dataset,
                                unprivileged_groups=[{'gender': 0}],  # female
                                privileged_groups=[{'gender': 1}])    # male
print("Original disparate impact:", metric.disparate_impact())

# Apply reweighing to mitigate bias
rw = Reweighing(unprivileged_groups=[{'gender': 0}],
               privileged_groups=[{'gender': 1}])
dataset_transformed = rw.fit_transform(dataset)

# Check bias after mitigation
metric_transformed = BinaryLabelDatasetMetric(dataset_transformed,
                                            unprivileged_groups=[{'gender': 0}],
                                            privileged_groups=[{'gender': 1}])
print("Transformed disparate impact:", metric_transformed.disparate_impact())

# Prepare debiased data for modeling
df_transformed = dataset_transformed.convert_to_dataframe()[0]
X = df_transformed[['years_experience', 'education_level']]  # Note: gender removed
y = df_transformed['hired']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train model on debiased data
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Check feature importance
print("Feature importances:", model.feature_importances_)
